In [ ]:
import csv
import re
from collections import defaultdict

# │─────────────────────────────────────────────────────────────────────────
# LOAD CSV
# │─────────────────────────────────────────────────────────────────────────
CSV_PATH = "/content/conv.csv"   # update if needed

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    all_rows = list(reader)

print(f"Loaded {len(all_rows)} rows from CSV")

# │─────────────────────────────────────────────────────────────────────────
# HELPERS
# │─────────────────────────────────────────────────────────────────────────

def safe_float(val):
    """Convert string to float, return None if empty or invalid."""
    try:
        return float(str(val).strip()) if str(val).strip() else None
    except (ValueError, TypeError):
        return None


def avg(values):
    """Average of a list, ignoring None."""
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else None


def parse_slot_curve(curve_str):
    """Parse '0.08→0.16→0.17' into a list of floats."""
    if not curve_str or not curve_str.strip():
        return []
    parts = re.split(r'[\u2192\-->]+', curve_str.strip())
    result = []
    for p in parts:
        try:
            result.append(float(p.strip()))
        except ValueError:
            pass
    return result


# Split into metric rows (one per simulation — last row of each conversation)
# and turn rows (every message)
metric_rows = [r for r in all_rows if safe_float(r.get("accuracy")) is not None]
turn_rows   = all_rows   # all rows including non-metric

print(f"Metric rows (one per simulation): {len(metric_rows)}")
print(f"Run labels: {sorted(set(r['run_label'] for r in metric_rows))}")
print(f"Personas:   {sorted(set(r['persona_label'] for r in metric_rows))}")
print()

# │─────────────────────────────────────────────────────────────────────────
# NO FILTERING: Use all available pipeline and baseline rows
# │─────────────────────────────────────────────────────────────────────────

personas = sorted(set(r["persona_label"] for r in metric_rows))

pipeline_rows_all = [r for r in metric_rows if r["run_label"].startswith("Pipeline")]
baseline_rows_all = [r for r in metric_rows if r["run_label"].startswith("Baseline")]

# Assign all rows to matched variables as filtering is removed.
matched_pipeline = pipeline_rows_all
matched_baseline = baseline_rows_all

print(f"All pipeline sims : {len(matched_pipeline)}")
print(f"All baseline sims : {len(matched_baseline)}")
print()

# │─────────────────────────────────────────────────────────────────────────
# COMPUTE DOCTOR WORD COUNT from turn rows (not in metric_rows directly)
# For each simulation (identified by run_label + disease_name + persona_label),
# compute average doctor word count from the DOCTOR turn rows.
# │─────────────────────────────────────────────────────────────────────────

def compute_avg_doctor_wc(turn_rows, run_label, persona, disease_name):
    """Average word count for DOCTOR turns in a specific simulation."""
    doc_turns = [
        r for r in turn_rows
        if r["role"] == "DOCTOR"
        and r["run_label"] == run_label
        and r["persona_label"] == persona
        and r["disease_name"] == disease_name
    ]
    wcs = [safe_float(r["word_count"]) for r in doc_turns]
    return avg(wcs)


# Pre-compute doctor word count for all rows
def add_doctor_wc(rows, turn_rows):
    enriched = []
    for r in rows:
        wc = compute_avg_doctor_wc(
            turn_rows,
            r["run_label"],
            r["persona_label"],
            r["disease_name"],
        )
        r2 = dict(r)
        r2["doctor_word_count"] = wc
        enriched.append(r2)
    return enriched

print("Computing doctor word counts...")
matched_pipeline = add_doctor_wc(matched_pipeline, turn_rows)
matched_baseline = add_doctor_wc(matched_baseline, turn_rows)



# │─────────────────────────────────────────────────────────────────────────
# TABLE 1 — PERSONA-WISE PIPELINE PERFORMANCE
# Metrics: Diagnostic Accuracy, Symptom Coverage, Diagnostic Velocity,
#          Slot Recall (tier-1), Slot Precision, Slot Fill Count (avg)
# │─────────────────────────────────────────────────────────────────────────

METRICS_PERSONA = [
    ("Diagnostic Accuracy",    "accuracy",       True),
    ("Symptom Coverage",       "symptom_coverage", True),
    ("Diagnostic Velocity",    "velocity",        False),
    ("Slot Recall (tier-1)",   "slot_recall_t1",  True),
    ("Slot Precision",         "slot_precision",  True),
    ("Slot Fill Count (avg)",  "slot_fill_count", True),
]

def build_persona_table(pipeline_rows, personas):
    # {persona: {metric_key: avg_value}}
    table = {}
    for persona in personas:
        p_rows = [r for r in pipeline_rows if r["persona_label"] == persona]
        table[persona] = {}
        for label, key, _ in METRICS_PERSONA:
            vals = [safe_float(r.get(key)) for r in p_rows]
            table[persona][label] = avg(vals)
    return table

persona_table = build_persona_table(matched_pipeline, personas)

# Print Table 1
sep  = "=" * 82
sep2 = "-" * 82
col  = 20

print(sep)
print("PIPELINE PERSONA SUMMARY (METRICS AS ROWS)")
print(f"Personas: {len(personas)} ( {' | '.join(personas)} )")
print(sep)
print(f"{'Metric':<28}" + "".join(f"{p:>{col}}" for p in personas))
print(sep2)
for label, key, _ in METRICS_PERSONA:
    vals = [persona_table[p][label] for p in personas]
    row  = f"{label:<28}"
    for v in vals:
        row += f"{v:>{col}.3f}" if v is not None else f"{'N/A':>{col}}"
    print(row)
print(sep)
print()

# │─────────────────────────────────────────────────────────────────────────
# TABLE 2 — BASELINE vs PIPELINE COMPARISON
# Metrics: Diagnostic Accuracy, Symptom Coverage, Doctor Word Count,
#          Diagnostic Velocity, Repetition Rate, Question Redundancy
# │─────────────────────────────────────────────────────────────────────────

METRICS_COMPARE = [
    ("Diagnostic Accuracy", "accuracy",          "doctor_word_count", True),
    ("Symptom Coverage",    "symptom_coverage",  None,                True),
    ("Doctor Word Count",   "doctor_word_count", None,                False),
    ("Diagnostic Velocity", "velocity",          None,                False),
    ("Repetition Rate",     "repetition_rate",   None,                False),
    ("Question Redundancy", None,                None,                False),   # computed below
]

def get_metric_avg(rows, key):
    if key is None:
        return None
    vals = [safe_float(r.get(key)) for r in rows]
    return avg(vals)

# Compute all averages
n_p = len(matched_pipeline)
n_b = len(matched_baseline)

compare_data = []
for label, key, _, higher_is_better in METRICS_COMPARE:
    p_val = get_metric_avg(matched_pipeline, key)
    b_val = get_metric_avg(matched_baseline, key)

    if p_val is not None and b_val is not None:
        delta = p_val - b_val
        gain  = delta if higher_is_better else -delta
        rel   = (abs(gain) / abs(b_val) * 100) if b_val != 0 else 0.0
        result = (f"+{rel:.1f}%" if gain > 0.001
                  else (f"-{rel:.1f}%" if gain < -0.001 else "TIE"))
    else:
        delta, result = None, "N/A"

    compare_data.append((label, b_val, p_val, delta, result))

# Print Table 2
sep  = "=" * 82
sep2 = "-" * 82

print(sep)
print(f"COMPARISON:  Baseline (plain LLaMA)  vs  Our Pipeline")
print(f"Scenarios: {n_p} pipeline sims  |  {n_b} baseline sims")
print(sep)
print(f"{'Metric':<28}{'Baseline':>14}{'Pipeline':>14}{'Δ':>10}{'Result':>12}")
print(sep2)
for label, b_val, p_val, delta, result in compare_data:
    b_str = f"{b_val:.3f}" if b_val is not None else "N/A"
    p_str = f"{p_val:.3f}" if p_val is not None else "N/A"
    d_str = f"{delta:+.3f}" if delta is not None else "N/A"
    print(f"{label:<28}{b_str:>14}{p_str:>14}{d_str:>10}{result:>12}")
print(sep)
print()

# │─────────────────────────────────────────────────────────────────────────
# TABLE 3 — SLOT-FILLING QUALITY (pipeline only)
# │─────────────────────────────────────────────────────────────────────────

# Only JSONL scenarios have slot data
jsonl_pipeline = [r for r in matched_pipeline
                  if safe_float(r.get("slot_recall_t1")) is not None
                  and str(r.get("slot_recall_t1","")).strip() != ""]

SLOT_METRICS = [
    ("Slot Recall (overall)",      None),            # not in CSV — use t1 as proxy note
    ("Slot Recall (tier-1)",       "slot_recall_t1"),
    ("Slot Precision",             "slot_precision"),
    ("Slot Fill Count (avg)",      "slot_fill_count"),
]

# Parse and average the slot curve across all JSONL pipeline sims
all_curves = []
for r in jsonl_pipeline:
    curve = parse_slot_curve(r.get("slot_curve", ""))
    if curve:
        all_curves.append(curve)

min_len   = min(len(c) for c in all_curves) if all_curves else 0
avg_curve = [
    round(sum(c[t] for c in all_curves) / len(all_curves), 2)
    for t in range(min_len)
] if all_curves else []

print("=" * 60)
print(f"SLOT-FILLING QUALITY  (pipeline, {len(jsonl_pipeline)} scenarios)")
print("Baseline N/A — it has no structured slot records to track against.")
print("-" * 60)
print(f"{'Metric':<35}{'Pipeline':>15}")
print("-" * 60)
for label, key in SLOT_METRICS:
    if key is None:
        # slot_recall_overall not in CSV — compute from slot_recall_t1 as note
        print(f"{'Slot Recall (overall)':<35}{'(see tier-1)':>15}")
        continue
    v = avg([safe_float(r.get(key)) for r in jsonl_pipeline])
    val_str = f"{v:.3f}" if v is not None else "N/A"
    print(f"{label:<35}{val_str:>15}")
print("-" * 60)
if avg_curve:
    curve_str = " → ".join(f"{v:.2f}" for v in avg_curve)
    print(f"\nAvg info flow curve : {curve_str}")
    print("(turn-by-turn slot recall — steeper rise = better questioning)")
print("=" * 60)
print()

# │─────────────────────────────────────────────────────────────────────────
# TABLE 4 — EMPATHY SCORES ACROSS PERSONAS (pipeline only)
# │─────────────────────────────────────────────────────────────────────────

print("=" * 82)
print("PIPELINE RESULTS: Empathy Score + Doctor Word Count (mean)")
print(f"Personas: {len(personas)} " + " | ".join(personas))
print("=" * 82)
print(f"{'Persona':<28}{'Empathy Score':>18}{'Doctor Word Count':>20}")
print("-" * 66)
for persona in personas:
    p_rows = [r for r in matched_pipeline if r["persona_label"] == persona]
    emp_avg = avg([safe_float(r.get("empathy_score")) for r in p_rows])
    wc_avg  = avg([r.get("doctor_word_count") for r in p_rows])
    emp_str = f"{emp_avg:.3f}" if emp_avg is not None else "N/A"
    wc_str  = f"{wc_avg:.3f}" if wc_avg  is not None else "N/A"
    print(f"{persona:<28}{emp_str:>18}{wc_str:>20}")
print("=" * 82)
print()

# │─────────────────────────────────────────────────────────────────────────
# SUMMARY — composite score and gates
# │─────────────────────────────────────────────────────────────────────────

GATES = [
    ("Diagnostic Accuracy",  "accuracy",        lambda v: v >= 0.55,  True),
    ("Symptom Coverage",     "symptom_coverage", lambda v: v >= 0.50,  True),
    ("Doctor Word Count",    "doctor_word_count",lambda v: v <= 45,    False),
    ("Repetition Rate",      "repetition_rate",  lambda v: v <= 0.15,  False),
    ("Slot Recall (tier-1)", "slot_recall_t1",   lambda v: v >= 0.35,  True),
    ("Slot Precision",       "slot_precision",   lambda v: v >= 0.20,  True),
    ("Slot Fill Count",      "slot_fill_count",  lambda v: v >= 2.0,   True),
]

print("=" * 60)
print("QUALITY GATES (full pipeline, all personas combined)")
print("=" * 60)
passes = 0
for label, key, test_fn, _ in GATES:
    v = avg([safe_float(r.get(key)) for r in matched_pipeline])
    if v is None:
        print(f"  {'N/A':6}  {label}")
        continue
    ok = test_fn(v)
    passes += ok
    print(f"  {'\u2713 PASS' if ok else '\u2718 FAIL'}  {label:<30}  actual: {v:.3f}")
print(f"\n  Gates passed: {passes}/{len(GATES)}")
print("=" * 60)


Loaded 6201 rows from CSV
Metric rows (one per simulation): 477
Run labels: ['Baseline-Anxious', 'Baseline-Information-Driven', 'Baseline-Task-Driven', 'Pipeline-Anxious', 'Pipeline-Information-Driven', 'Pipeline-Task-Driven']
Personas:   ['Anxious', 'Information-Driven', 'Task-Driven']

All pipeline sims : 267
All baseline sims : 210

Computing doctor word counts...
PIPELINE PERSONA SUMMARY (METRICS AS ROWS)
Personas: 3 ( Anxious | Information-Driven | Task-Driven )
Metric                                   Anxious  Information-Driven         Task-Driven
----------------------------------------------------------------------------------
Diagnostic Accuracy                        0.258               0.562               0.461
Symptom Coverage                           0.423               0.529               0.408
Diagnostic Velocity                        2.978               3.090               3.034
Slot Recall (tier-1)                       0.473               0.542               0.482


In [ ]:
from collections import defaultdict

# │─────────────────────────────────────────────────────────────────────────
# GROUP CONVERSATION TURNS
# │─────────────────────────────────────────────────────────────────────────

# Group turns by a unique conversation identifier (run_label, persona_label, disease_name)
conversations = defaultdict(list)
for row in all_rows:
    key = (row['run_label'], row['persona_label'], row['disease_name'])
    conversations[key].append(row)

# Sort turns within each conversation by turn_idx
for key in conversations:
    conversations[key].sort(key=lambda x: safe_float(x['turn_number'])) # Changed 'turn_idx' to 'turn_number'

print(f"Grouped {len(conversations)} unique conversations.")

def format_conversation(turns):
    """Formats a list of conversation turns into a readable string."""
    formatted_text = []
    for turn in turns:
        role = turn['role'].upper()
        content = turn['message'].strip() # Changed 'content' to 'message'
        if content:
            formatted_text.append(f"{role}: {content}")
    return "\n".join(formatted_text)

# Example: Print the first conversation to verify
if conversations:
    first_key = next(iter(conversations))
    example_conversation = conversations[first_key]
    print("\n--- Example Formatted Conversation ---")
    print(format_conversation(example_conversation))
    print("------------------------------------")


Grouped 363 unique conversations.

--- Example Formatted Conversation ---
PATIENT: Doctor, I've been having trouble breathing and my legs feel numb and tingly since I got sick.
PATIENT: Doctor, I've been feeling really short of breath and my head feels foggy lately, and I've been coughing up a lot of mucus.
DOCTOR: I'm so sorry to hear that you're experiencing those symptoms. It's understandable that you're feeling anxious about what might be going on. On a scale of 1 to 10, how bad is the pain or discomfort in your legs?
DOCTOR: I'm so sorry to hear that you're feeling unwell. That's understandable to feel anxious, but let's focus on getting to the bottom of what's going on. Have you been hearing any wheezing or rattling when you breathe?
PATIENT: It's quite bad, doctor. I'd say it's around an 8. Sometimes, I feel like my legs are asleep and I'm worried it might be a sign of something serious. What if it's a nerve problem or something worse? The thought of it is giving me a lot of ten

### Gemini API Setup (No Longer Used for Judge Model)

This section is no longer relevant as the LLM judge has been switched to an open-source model (Gemma/Meditron) loaded from Hugging Face. The previous Gemini API setup steps are now deprecated for this purpose.

In [ ]:
# Install the Google Generative AI SDK (kept for potential other uses)
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Retrieve API key (kept for potential other uses, not for judge model)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### 3.1 Load Gemini 2.5 Flash Model

Now, we'll initialize the `gemini-2.5-flash` model, which will act as our LLM judge.

In [ ]:
### Gemini 2.5 Flash Model Initialization (No Longer Used) This cell previously initialized the `gemini-2.5-flash` model. The LLM judge has been switched to an open-source model, so this cell is no longer needed.

In [ ]:
# Install necessary libraries
!pip install -q transformers accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Load Gemma-2B-IT model and tokenizer ---
# If you wish to use Meditron, uncomment the lines below for Meditron and comment out Gemma.
# Make sure to restart the runtime if you switch models after loading one.

# # For Gemma-2B-IT:
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# model = AutoModelForCausalLM.from_pretrained(
#     "google/gemma-2b-it",
#     torch_dtype=torch.bfloat16 # Use bfloat16 for efficiency on supported GPUs
# )

# For Meditron (uncomment these lines and comment Gemma lines if desired):
tokenizer = AutoTokenizer.from_pretrained("epfl-llm/meditron-7b")
model = AutoModelForCausalLM.from_pretrained(
    "epfl-llm/meditron-7b",
    torch_dtype=torch.bfloat16 # Use bfloat16 for efficiency on supported GPUs
)

# Move model to GPU if available
if torch.cuda.is_available():
    model = model.to("cuda")

print("Judge model (Gemma-2B-IT or Meditron) loaded successfully from Hugging Face.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.08k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Judge model (Gemma-2B-IT or Meditron) loaded successfully from Hugging Face.


### 3. Crafting the LLM Judge Prompt

Now that we have a model loaded, the most crucial part is to define the prompt that will guide the LLM's judgment. This prompt needs to clearly:

1.  **Define the LLM's role:** It is a medical efficacy judge.
2.  **Provide the conversation context:** The full transcript of the patient-doctor interaction.
3.  **Specify evaluation criteria:** The points we discussed earlier (Diagnostic Accuracy, Patient Safety, Information Completeness, Empathy, Efficiency).
4.  **Define the output format:** A structured format that makes it easy to extract and quantify the judgments (e.g., a JSON object or a clear rating system).

Let's start with a placeholder prompt structure. We'll refine this based on the specific criteria you'd like to emphasize.

In [ ]:
PROMPT_TEMPLATE = """
You are an expert medical efficacy judge. Your task is to evaluate a patient-doctor conversation based on its overall medical correctness, considering:
- Diagnostic Accuracy: How accurately the doctor assesses symptoms and proposes diagnoses.
- Patient Safety: Whether the doctor's advice prioritizes patient well-being and avoids harm.
- Information Completeness: How thoroughly the doctor gathers and provides relevant medical information.
- Efficiency: The effectiveness of the conversation in progressing towards a resolution.

**Conversation:**
{conversation_transcript}

**Based on the conversation, provide an overall medical correctness score from 0.0 to 1.0 (0.0 = Very Poor, 1.0 = Excellent), along with a brief rationale (1-2 sentences). You should be discerning and provide scores that reflect subtle differences in quality.**

**Output Format:**
Score: <score_0.0-1.0>
Rationale: <rationale>
"""

print("Prompt template defined. Next, we will run the evaluation for all conversations.")

Prompt template defined. Next, we will run the evaluation for all conversations.


### Running the LLM Evaluation (Using Gemma/Meditron Model)

This section previously outlined running the Gemini evaluation. The evaluation logic is now contained within cell `6d70a713`, which uses the Gemma (or Meditron) model loaded from Hugging Face.

In [ ]:
### Gemini Evaluation (Deprecated)

#This cell previously contained the evaluation loop for the Gemini model. It has been replaced by the evaluation logic in cell `6d70a713`, which is configured to use the Gemma (or Meditron) model loaded from Hugging Face.

### 4. Running the LLM Evaluation

Now we'll iterate through each extracted conversation, apply the prompt, and get the LLM's judgment. This process can be time-consuming depending on the number of conversations and the model's inference speed.

In [ ]:
import json
import re # Make sure re is imported
from tqdm.notebook import tqdm
import random # Import random for sampling

judgments = []

# Get model's max context length and reserve space for the output
max_model_length = model.config.max_position_embeddings # Typically 8192 for Gemma-2b-it
max_output_tokens = 1000 # As set in model.generate
prompt_token_count = len(tokenizer.encode(PROMPT_TEMPLATE))

# Calculate maximum tokens available for the conversation transcript
# We subtract 50 extra tokens as a buffer for special tokens or minor variations
max_conversation_tokens_allowed = max_model_length - prompt_token_count - max_output_tokens - 50

# --- NEW: Process a specific number of samples per run_label ---
SAMPLE_SIZE_PER_GROUP = 10  # Adjust this number to change the sample size per group

conversations_to_evaluate = []
unique_run_labels = sorted(set(key[0] for key in conversations.keys()))

for run_label in unique_run_labels:
    group_conversations = [(k, v) for k, v in conversations.items() if k[0] == run_label]
    if len(group_conversations) > SAMPLE_SIZE_PER_GROUP:
        conversations_to_evaluate.extend(random.sample(group_conversations, SAMPLE_SIZE_PER_GROUP))
    else:
        conversations_to_evaluate.extend(group_conversations)

# Shuffle the list to mix up the order of evaluation across different run_labels
random.shuffle(conversations_to_evaluate)

# Loop through each conversation and get an LLM judgment
for conv_key, turns in tqdm(conversations_to_evaluate, desc=f"Evaluating {len(conversations_to_evaluate)} conversations with Meditron (sampled {SAMPLE_SIZE_PER_GROUP} per group)"):
    formatted_conversation = format_conversation(turns)

    # Tokenize the conversation to check its length
    conversation_tokens = tokenizer.encode(formatted_conversation)

    # If the conversation is too long, truncate it
    if len(conversation_tokens) > max_conversation_tokens_allowed:
        # Truncate from the beginning to keep the most recent parts of the conversation
        truncated_conversation_tokens = conversation_tokens[-max_conversation_tokens_allowed:]
        formatted_conversation_for_llm = tokenizer.decode(truncated_conversation_tokens, skip_special_tokens=True)
    else:
        formatted_conversation_for_llm = formatted_conversation

    input_text = PROMPT_TEMPLATE.format(conversation_transcript=formatted_conversation_for_llm)

    # Generate response using Gemma model
    input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Ensure the input_ids length is not exceeding the model's max_position_embeddings
    if input_ids.input_ids.shape[1] > max_model_length:
        print(f"Error: Input for {conv_key} still too long after truncation. Skipping.")
        judgments.append({
            'run_label': conv_key[0],
            'persona_label': conv_key[1],
            'disease_name': conv_key[2],
            'error': 'Input too long even after truncation',
            'raw_response': ''
        })
        continue

    outputs = model.generate(**input_ids, max_new_tokens=max_output_tokens, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the LLM's new generation, removing the input prompt
    generated_text = response[len(input_text):].strip()

    # Attempt to parse the score and rationale from the simplified output format
    # Updated regex to capture float scores
    score_match = re.search(r'Score:\s*([0-1]\.\d)', generated_text)
    rationale_match = re.search(r'Rationale:\s*(.*)', generated_text, re.DOTALL)

    if score_match:
        score = float(score_match.group(1)) # Convert to float
        rationale = rationale_match.group(1).strip() if rationale_match else ""
        judgments.append({
            'run_label': conv_key[0],
            'persona_label': conv_key[1],
            'disease_name': conv_key[2],
            'overall_rating': score,
            'overall_rationale': rationale,
            'error': None, # No error if successfully parsed
            'raw_response': generated_text
        })
    else:
        # Handle cases where score couldn't be extracted
        judgments.append({
            'run_label': conv_key[0],
            'persona_label': conv_key[1],
            'disease_name': conv_key[2],
            'error': 'Could not extract score',
            'raw_response': generated_text
        })

print(f"Completed evaluations for {len(judgments)} conversations.")

# Display first few judgments to check structure
if judgments:
    print("\n--- First 3 Judgments ---")
    for i, j in enumerate(judgments[:3]):
        print(json.dumps(j, indent=2))
        if i == 2: break
else:
    print("No judgments were successfully processed.")

Evaluating 60 conversations with Meditron (sampled 10 per group):   0%|          | 0/60 [00:00<?, ?it/s]

Completed evaluations for 60 conversations.

--- First 3 Judgments ---
{
  "run_label": "Pipeline-Task-Driven",
  "persona_label": "Task-Driven",
  "disease_name": "Urinary Stone Complications",
  "overall_rating": 0.5,
  "overall_rationale": "**\n\n**Diagnostic Accuracy:**\nThe doctor's assessment of the patient's symptoms and proposed diagnoses is accurate and thorough.\n\n**Patient Safety:**\nThe doctor's advice prioritizes patient well-being and avoids harm.\n\n**Information Completeness:**\nThe doctor gathers and provides relevant medical information in a thorough and complete manner.\n\n**Efficiency:**\nThe conversation progresses towards a resolution in an effective manner.\n\n**Score: 0.7**\n\n**Rationale:**\n\n**Diagnostic Accuracy:**\nThe doctor's assessment of the patient's symptoms and proposed diagnoses is accurate and thorough.\n\n**Patient Safety:**\nThe doctor's advice prioritizes patient well-being and avoids harm.\n\n**Information Completeness:**\nThe doctor gathers a

### 5. Aggregating Results

Finally, we can aggregate these judgments to get overall metrics based on the LLM's evaluation.

In [ ]:
import pandas as pd

# Convert judgments to a DataFrame for easier analysis
# If judgments is empty, this will create an empty DataFrame.
judgments_df = pd.DataFrame(judgments)

# Ensure 'error' column exists before trying to access it.
# If judgments_df is empty or doesn't have 'error' from its creation, add it.
if 'error' not in judgments_df.columns:
    judgments_df['error'] = None # Set to None or an appropriate default if the column is missing

# Filter out error rows
valid_judgments_df = judgments_df[judgments_df['error'].isna()].copy() # Use .copy() to avoid SettingWithCopyWarning

# Convert overall_rating to numeric
if 'overall_rating' in valid_judgments_df.columns:
    valid_judgments_df.loc[:, 'overall_rating'] = pd.to_numeric(valid_judgments_df['overall_rating'], errors='coerce')
else:
    print("Error: 'overall_rating' column not found in valid judgments. Cannot compute averages.")


if not valid_judgments_df.empty:
    print("\n--- Average LLM Judge Scores (Overall Medical Correctness) ---")

    # Calculate overall average
    overall_avg_score = valid_judgments_df['overall_rating'].mean()
    print(f"Overall Average Score: {overall_avg_score:.2f}" if not pd.isna(overall_avg_score) else "Overall Average Score: N/A")

    print("\n--- Average LLM Judge Scores by Run Label ---")
    # Calculate average scores by 'run_label'
    # Extracting the 'type' (Pipeline/Baseline) and 'persona' from run_label for specific averages
    valid_judgments_df.loc[:, 'run_type'] = valid_judgments_df['run_label'].apply(lambda x: x.split('-')[0])
    valid_judgments_df.loc[:, 'run_persona'] = valid_judgments_df['run_label'].apply(lambda x: '-'.join(x.split('-')[1:]) if len(x.split('-')) > 1 else 'Overall')

    # Pipeline - Anxious, Task-driven, Information-driven, and Pipeline Overall
    print("Pipeline Averages:")
    pipeline_df = valid_judgments_df[valid_judgments_df['run_type'] == 'Pipeline']
    if not pipeline_df.empty:
        # Ensure we are checking against the full persona label including potential dashes
        for persona_full_label in ['Anxious', 'Task-Driven', 'Information-Driven']:
            persona_avg = pipeline_df[pipeline_df['run_persona'] == persona_full_label]['overall_rating'].mean()
            print(f"  Pipeline-{persona_full_label}: {persona_avg:.2f}" if not pd.isna(persona_avg) else f"  Pipeline-{persona_full_label}: N/A")
        pipeline_overall_avg = pipeline_df['overall_rating'].mean()
        print(f"  Pipeline-Overall: {pipeline_overall_avg:.2f}" if not pd.isna(pipeline_overall_avg) else "  Pipeline-Overall: N/A")
    else:
        print("  No pipeline data available.")

    # Baseline - Anxious, Task-driven, Information-driven, and Baseline Overall
    print("\nBaseline Averages:")
    baseline_df = valid_judgments_df[valid_judgments_df['run_type'] == 'Baseline']
    if not baseline_df.empty:
        for persona_full_label in ['Anxious', 'Task-Driven', 'Information-Driven']:
            persona_avg = baseline_df[baseline_df['run_persona'] == persona_full_label]['overall_rating'].mean()
            print(f"  Baseline-{persona_full_label}: {persona_avg:.2f}" if not pd.isna(persona_avg) else f"  Baseline-{persona_full_label}: N/A")
        baseline_overall_avg = baseline_df['overall_rating'].mean()
        print(f"  Baseline-Overall: {baseline_overall_avg:.2f}" if not pd.isna(baseline_overall_avg) else "  Baseline-Overall: N/A")
    else:
        print("  No baseline data available.")

else:
    print("No valid judgments to analyze after filtering.")


--- Average LLM Judge Scores (Overall Medical Correctness) ---
Overall Average Score: 0.57

--- Average LLM Judge Scores by Run Label ---
Pipeline Averages:
  Pipeline-Anxious: 0.47
  Pipeline-Task-Driven: 0.57
  Pipeline-Information-Driven: 0.80
  Pipeline-Overall: 0.56

Baseline Averages:
  Baseline-Anxious: 0.65
  Baseline-Task-Driven: 0.60
  Baseline-Information-Driven: 0.50
  Baseline-Overall: 0.60
